# DESlab en Python 3 / Google Colab

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nebisman/deslab-py3/blob/main/notebooks/DESlab_Colab.ipynb)

DESlab es una herramienta para el análisis y la síntesis de **sistemas a eventos discretos** modelados como autómatas
(L. Bermeo, J. C. Basilio). Esta versión funciona con Python 3 y las versiones actuales de `networkx` y `graphviz`.
Cada llamada a `draw(...)` muestra los autómatas **directamente en el notebook** (no se generan PDF).

Repositorio: https://github.com/nebisman/deslab-py3

**Cómo usarlo en Colab:** ejecute primero la celda de instalación (instala DESlab desde GitHub) y luego las demás
celdas en orden, o use *Entorno de ejecución → Ejecutar todas*. Para guardar sus cambios use
*Archivo → Guardar una copia en Drive*.

In [ ]:
#@title Instalación de DESlab desde GitHub (ejecute esta celda primero)
import shutil, subprocess

# programas de graphviz (Colab normalmente ya los trae instalados)
if shutil.which("dot") is None:
    subprocess.run("apt-get -qq install -y graphviz > /dev/null", shell=True)

%pip install -q git+https://github.com/nebisman/deslab-py3.git

In [ ]:
from deslab import *

## 1. Definición de un autómata

`syms` crea variables cuyo valor es su propio nombre. La tabla asocia a cada símbolo una etiqueta LaTeX
que se usa en los dibujos. Los eventos no observables se dibujan con líneas punteadas.

In [ ]:
syms('q1 q2 q3 a1 b1 e')
table = [(a1, '\\alpha_1'), (b1, '\\beta_1'), (q1, 'q_1'), (q2, 'q_2'), (q3, 'q_3')]
X = [q1, q2, q3]
Sigma = [a1, b1, e]
X0 = [q1]
Xm = [q1, q3]
T = [(q1, b1, q2), (q2, b1, q3), (q2, e, q3), (q3, a1, q1), (q2, a1, q2)]
G1 = fsa(X, Sigma, T, X0, Xm, table, Sigobs=[a1, b1], name='$G_1$ -- ejemplo 1')
draw(G1)

Un autómata en la última línea de una celda también se dibuja automáticamente:

In [ ]:
G1

In [ ]:
print(G1)
print('Estados:', G1.X, ' Eventos:', G1.Sigma)
print('Gamma(q2) =', G1.Gamma(q2), '   delta(q1, b1) =', G1.delta(q1, b1))
print('¿Es determinista?', G1.is_dfa())

## 2. Operaciones

| Operación | Notación |
|---|---|
| Producto | `G1 & G2` |
| Composición paralela | `G1 // G2` |
| Unión de lenguajes | `G1 + G2` |
| Concatenación | `G1 * G2` |
| Complemento | `~G1` |
| Diferencia | `G1 - G2` |
| Cociente | `G1 / G2` |
| Clausura de Kleene | `kleeneclos(G1)` |
| Parte accesible, coaccesible, trim | `ac(G)`, `coac(G)`, `trim(G)` |

In [ ]:
syms('a b c d e f s1 s2 s3 s4')
tab = [(a, '\\alpha'), (b, '\\beta'), (s1, 'x_1'), (s2, 'x_2'), (s3, 'x_3'), (s4, 'x_4')]
H1 = fsa([s1, s2, s3, s4], [a, b, c], [(s1, a, s2), (s2, b, s3), (s3, c, s4), (s1, b, s4)],
         [s1], [s2, s3, s4], tab, name='$H_1$')
H2 = fsa([s1, s2, s3, s4], [a, b, c, d, e, f],
         [(s1, a, s2), (s2, e, s3), (s3, f, s4), (s1, b, s3), (s1, f, s4), (s2, f, s4)],
         [s1], [s1, s2], tab, name='$H_2$')
draw(H1, H2,
     (H1 & H2).setpar(name='$H_1 \\times H_2$'),
     (H1 // H2).setpar(name='$H_1 \\| H_2$'))

In [ ]:
draw((H1 + H2).setpar(name='$L_m(H_1) \\cup L_m(H_2)$'),
     (~H1).setpar(name='$L_m(H_1)^c$'),
     kleeneclos(H1).setpar(name='$L_m(H_1)^*$'))

## 3. Observador y proyección

`observer(G, Sigma_o)` calcula el autómata determinista que genera y marca la proyección de los lenguajes
de `G` sobre los eventos observables `Sigma_o` (por defecto `G.Sigobs`).

In [ ]:
syms('a b c d ed u v g')
syms('s1 s2 s3 s4 s5 s6 s7 s8 s9 s10 s11 s12')
T = [(s1, ed, s2), (s1, a, s4), (s2, u, s3), (s2, a, s5), (s3, c, s7), (s4, ed, s5),
     (s4, b, s8), (s5, u, s6), (s5, b, s9), (s6, g, s7), (s7, v, s5), (s7, b, s10),
     (s8, d, s1), (s8, ed, s9), (s9, u, s10), (s9, d, s2), (s10, d, s3), (s8, v, s12),
     (s12, ed, s11), (s11, d, s1)]
tab = [(ed, 'e_d')] + [(s, 's_{%s}' % s[1:]) for s in [s1, s2, s3, s4, s5, s6, s7, s8, s9, s10, s11, s12]]
G = fsa([s1, s2, s3, s4, s5, s6, s7, s8, s9, s10, s11, s12], [a, b, c, d, ed, u, v, g], T,
        [s1], [s1], tab, Sigobs=[a, b, c, d, g], name='$G$')
draw(G, observer(G).setpar(name='$Obs(G)$'))

## 4. Diagnóstico de fallas

El diagnosticador es el observador de $G \| A_l$, donde $A_l$ es el autómata rotulador de la falla
(`N`: comportamiento normal, `Y`: ocurrió la falla).

In [ ]:
syms('a b c d e sf x1 x2 x3 x4 x5 x6 x7')
X = [x1, x2, x3, x4, x5, x6, x7]
T = [(x1, c, x2), (x1, a, x5), (x2, sf, x3), (x3, e, x4), (x4, d, x4),
     (x5, b, x6), (x6, d, x6), (x7, e, x7), (x3, a, x7)]
tab = [(x, 'x_%s' % x[1:]) for x in X] + [(sf, '\\sigma_f')]
G = fsa(X, [a, b, c, d, e, sf], T, [x1], [], tab, Sigobs=[a, b, c, d, e], name='$G$')
draw(G, diagnoser(G, sf))

## 5. Control supervisor

Fábrica con dos máquinas y una unidad de prueba (ejemplo `example_supervisorio.py` del DESlab original).
`supcontrollable(H, G)` calcula el autómata que marca el supremo sublenguaje controlable de la especificación.

In [ ]:
syms('x0 x1 y0 y1 z0 z1 z2 z3 xh10 xh11 xh20 xh21 e1 e2 e3 e4 e5 e60 e80 e62 e82')
tab = [(e1, 'e_1'), (e2, 'e_2'), (e3, 'e_3'), (e4, 'e_4'), (e5, 'e_5'),
       (e60, 'e_{60}'), (e62, 'e_{62}'), (e80, 'e_{80}'), (e82, 'e_{82}')]
GM1 = fsa([x0, x1], [e1, e2], [(x0, e1, x1), (x1, e2, x0)], [x0], [x0], tab, name='$G_{M1}$')
GM2 = fsa([y0, y1], [e3, e4], [(y0, e3, y1), (y1, e4, y0)], [y0], [y0], tab, name='$G_{M2}$')
Gut = fsa([z0, z1, z2, z3], [e5, e62, e82, e80, e60],
          [(z0, e5, z1), (z1, e60, z2), (z2, e62, z0), (z1, e80, z3), (z3, e82, z0)],
          [z0], [z0], tab, name='$G_{ut}$')
Hs1 = fsa([xh10, xh11], [e2, e82, e3], [(xh10, e2, xh11), (xh10, e82, xh11), (xh11, e3, xh10)],
          [xh10], [xh10, xh11], tab, name='$H_{spec,1}$', type='specification')
Hs2 = fsa([xh20, xh21], [e4, e5], [(xh20, e4, xh21), (xh21, e5, xh20)],
          [xh20], [xh20, xh21], tab, name='$H_{spec,2}$', type='specification')
draw(GM1, GM2, Gut, Hs1, Hs2)

In [ ]:
Gf = (GM1 // GM2 // Gut).setpar(Sigcon=[e1, e3, e5])   # e1, e3, e5 controlables
R = (Gf // Hs1 // Hs2).renamestates('number')
Gf = Gf.renamestates('number')
Ksup = supcontrollable(R, Gf).setpar(type='supervisor')
print('¿R es controlable?', iscontrollable(R, Gf), '  ¿Ksup es controlable?', iscontrollable(Ksup, Gf))
draw(Ksup)

## 6. Comparación de lenguajes

In [ ]:
print('Lm(H1 x H2) ⊆ Lm(H1):', (H1 & H2) <= H1)
print('Lm(H1) = Lm(~~H1):  ', are_langequiv(H1, ~~H1))
print('Lm(H1) vacío:       ', isitempty(H1))

## 7. Estilos y exportación

* `draw(G, style='figure')`: blanco y negro sin título; `'figurecolor'`: con colores sin título.
* `savefig(G, 'G.pdf')` guarda la figura (`pdf`, `png`, `svg`, `dot`, `tex`).
* `automaton2tikz(G)` devuelve el código TikZ y `drawlatex(G1, G2, style='beamer')` genera el PDF con LaTeX
  como el DESlab original (en Colab requiere instalar LaTeX: `!apt-get install -y texlive-latex-extra`).

In [ ]:
draw(G1, style='figure')
savefig(G1, 'G1.pdf')
print(automaton2tikz(G1))